In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# RGB-DCT receiver: fixed old-OFF development diagnostic

Run all once. This only reads eight already seen historical OFF FULL.mp4 files, checks each byte SHA-256, and applies the new uncalibrated receiver to matching files. Historical calibration/evaluation labels are report labels, not new calibration or independent evaluation. Old marked videos are not H1 for this receiver. There is no threshold, PASS, FPR, or TPR conclusion. The user runs this CPU notebook; it does not install model packages or run a model.


In [ ]:
from pathlib import Path
import datetime, json, sys
SOURCE_SHA = 'c967db03767768871ba0cd3b49208aa308f51b48'
INPUT_ROOT = Path('/content/drive/MyDrive/Video-WM/Content-Background-Existence-V1/content_background_existence_v1_20260923T024543022721Z')
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/RGB-DCT-Receiver-Diagnostic-V1')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / stamp
OUTPUT.mkdir(exist_ok=False)
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(status='SETUP_STARTED', source_sha=SOURCE_SHA, input_root=str(INPUT_ROOT), output_dir=str(OUTPUT), python=sys.version, executable=sys.executable), indent=2) + '\n', encoding='utf-8')
print('fixed input:', INPUT_ROOT, flush=True)
print('fresh output:', OUTPUT, flush=True)


In [ ]:
import subprocess, sys, json
SETUP_LOG = OUTPUT / 'setup.log'
def logged_run(command, *, cwd=None, check=True):
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True); log.write(line); log.flush()
        child = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in child.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n'); log.flush()
    if check and returncode:
        (OUTPUT / 'setup_failure.json').write_text(json.dumps(dict(command=command, returncode=returncode), indent=2) + '\n', encoding='utf-8')
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)


In [ ]:
REPO = Path('/content/SC-SSTW-RGB-DCT-' + stamp)
logged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])
logged_run(['git', '-C', str(REPO), 'fetch', 'origin', 'dev/rgb-dct-temporal-balanced-v1'])
logged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError('immutable source SHA readback mismatch')
(OUTPUT / 'source_receipt.json').write_text(json.dumps(dict(expected_sha=SOURCE_SHA, actual_sha=actual, repo=str(REPO)), indent=2) + '\n', encoding='utf-8')
print('source commit:', actual, flush=True)


In [ ]:
import shutil, numpy, torch
cpu_probe = torch.tensor([1.0], device='cpu')
environment_receipt = dict(ffmpeg=shutil.which('ffmpeg'), ffprobe=shutil.which('ffprobe'), numpy=numpy.__version__, torch=torch.__version__, torch_cpu_available=(cpu_probe.item() == 1.0))
(OUTPUT / 'environment_receipt.json').write_text(json.dumps(environment_receipt, indent=2) + '\n', encoding='utf-8')
if not environment_receipt['ffmpeg'] or not environment_receipt['ffprobe'] or not environment_receipt['torch_cpu_available']:
    raise RuntimeError('ffmpeg, ffprobe, and torch CPU are required for fixed RGB24 MP4 readback')
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(status='SETUP_COMPLETE', source_sha=SOURCE_SHA, input_root=str(INPUT_ROOT), output_dir=str(OUTPUT), python=sys.version, executable=sys.executable), indent=2) + '\n', encoding='utf-8')
print('CPU readback environment:', environment_receipt, flush=True)


In [ ]:
command = [sys.executable, '-u', '-m', 'scripts.rgb_dct_off_diagnostic', '--run-root', str(INPUT_ROOT), '--output', str(OUTPUT)]
completed = logged_run(command, cwd=REPO, check=False)
RESULT_PATH = OUTPUT / 'result.json'
(OUTPUT / 'execution_receipt.json').write_text(json.dumps(dict(command=command, returncode=completed.returncode, result_path=str(RESULT_PATH)), indent=2) + '\n', encoding='utf-8')
if not RESULT_PATH.exists():
    raise FileNotFoundError('runner produced no retained result.json')


In [ ]:
result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
if result['fixed_denominator'] != 8 or len(result['rows']) != 8:
    raise RuntimeError('fixed eight-row result missing')
print('scored:', result['scored_count'], 'invalid:', result['invalid_count'], 'pending:', result['pending_count'], flush=True)
print('full result:', RESULT_PATH, flush=True)
